## Website traffic analysis 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [2]:
df = pd.read_csv('traffic.csv')
df.head()


,event,date,country,city,artist,album,track,isrc,linkid
0,click,2021-08-21,Saudi Arabia,Jeddah,Tesher,Jalebi Baby,Jalebi Baby,QZNWQ2070741,2d896d31-97b6-4869-967b-1c5fb9cd4bb8
1,click,2021-08-21,Saudi Arabia,Jeddah,Tesher,Jalebi Baby,Jalebi Baby,QZNWQ2070741,2d896d31-97b6-4869-967b-1c5fb9cd4bb8
2,click,2021-08-21,India,Ludhiana,Reyanna Maria,So Pretty,So Pretty,USUM72100871,23199824-9cf5-4b98-942a-34965c3b0cc2
3,click,2021-08-21,France,Unknown,"Simone & Simaria, Sebastian Yatra",No Llores Más,No Llores Más,BRUM72003904,35573248-4e49-47c7-af80-08a960fa74cd
4,click,2021-08-21,Maldives,Malé,Tesher,Jalebi Baby,Jalebi Baby,QZNWQ2070741,2d896d31-97b6-4869-967b-1c5fb9cd4bb8


In [3]:
df['date'] = pd.to_datetime(df['date'])

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226278 entries, 0 to 226277
Data columns (total 9 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   event    226278 non-null  object        
 1   date     226278 non-null  datetime64[ns]
 2   country  226267 non-null  object        
 3   city     226267 non-null  object        
 4   artist   226241 non-null  object        
 5   album    226273 non-null  object        
 6   track    226273 non-null  object        
 7   isrc     219157 non-null  object        
 8   linkid   226278 non-null  object        
dtypes: datetime64[ns](1), object(8)
memory usage: 15.5+ MB


In [5]:
df.shape

(226278, 9)

In [6]:
df.isnull().sum()

event         0
date          0
country      11
city         11
artist       37
album         5
track         5
isrc       7121
linkid        0
dtype: int64

## Question 1
### [Pandas] How many total pageview events did the links in the provided dataset receive in the full period, how many per day?



In [7]:
df['event'].value_counts()

event
pageview    142015
click        55732
preview      28531
Name: count, dtype: int64

In [8]:
df[df['event'] == 'pageview'].groupby('date')["event"].count()

date
2021-08-19    22366
2021-08-20    21382
2021-08-21    21349
2021-08-22    20430
2021-08-23    18646
2021-08-24    18693
2021-08-25    19149
Name: event, dtype: int64

## Question 2
### [Pandas] What about the other recorded events?

In [9]:
other_events = set(df['event'].unique())
other_events.remove('pageview')
other_events

{'click', 'preview'}

In [10]:
for other_event in other_events:
    total_events = df[df['event'] == other_event].shape[0]
    print("Total number of", other_event, "events is", total_events, "\n")
    temp_df = df[df['event']== other_event].groupby('date')['event'].count()
    print(other_event, "event distribution per day:\n")
    print(temp_df, "\n")

    

Total number of preview events is 28531 

preview event distribution per day:

date
2021-08-19    3788
2021-08-20    4222
2021-08-21    4663
2021-08-22    4349
2021-08-23    3847
2021-08-24    3840
2021-08-25    3822
Name: event, dtype: int64 

Total number of click events is 55732 

click event distribution per day:

date
2021-08-19    9207
2021-08-20    8508
2021-08-21    8071
2021-08-22    7854
2021-08-23    7315
2021-08-24    7301
2021-08-25    7476
Name: event, dtype: int64 



## Question 3
### [Pandas] Which countries did the pageviews come from?

In [11]:
df_click_countries = df[df['event'] == 'pageview']

In [12]:
pd.DataFrame(data=df_click_countries['country'].dropna().unique(), columns=['Country'])


,Country
0,Saudi Arabia
1,United States
2,Ireland
3,United Kingdom
4,France
...,...
206,Afghanistan
207,Central African Republic
208,Guernsey
209,Sint Maarten


## Question 4
### [Pandas] What was the overall click rate (clicks/pageviews)?

**Calculating overall click rate**

In [13]:
all_clicks = df[df['event'] == 'click']['event'].count()
print("All clicks: ", all_clicks)
all_pageviews = df[df['event'] == 'pageview']['event'].count()
print("All pageviews: ", all_pageviews)

All clicks:  55732
All pageviews:  142015


In [14]:
ctr = round(all_clicks/all_pageviews,2)
print("Overlall CTR: ", ctr)

Overlall CTR:  0.39


### Calculating CTR per unique link

In [19]:
link_clicks = (
    df[df['event'] == 'click']
    .groupby('linkid')
    .size()
    .reset_index(name='clicks')
)

In [20]:
link_clicks

,linkid,clicks
0,00126b32-0c35-507b-981c-02c80d2aa8e7,2
1,004b9724-abca-5481-b6e9-6148a7ca00a5,1
2,0063a982-41cd-5629-96d0-e1c4dd72ea11,2
3,006af6a0-1f0d-4b0c-93bf-756af9071c06,8
4,00759b81-3f04-4a61-b934-f8fb3185f4a0,3
...,...,...
2250,ffd8d5a7-91bc-48e1-a692-c26fca8a8ead,29
2251,fff38ca0-8043-50cd-a5f1-f65ebb7105c5,1
2252,fff84c0e-90a1-59d8-9997-adc909d50e16,1
2253,fffc17a7-f935-5d3e-bd3e-d761fd80d479,1


In [21]:
link_pageviews = (
    df[df['event'] == 'pageview']
    .groupby('linkid')
    .size()
    .reset_index(name='page_views')
)

In [22]:
link_pageviews

,linkid,page_views
0,00073307-ae96-5089-a117-4783afb42f8e,2
1,00126b32-0c35-507b-981c-02c80d2aa8e7,2
2,0018cfff-50a1-5984-9715-01ef2d11a49a,1
3,0033934b-5d16-5a06-af58-d087bcdd3680,1
4,0034d6cf-3bd8-5ffe-aafc-b3959fc48608,1
...,...,...
3832,fff38ca0-8043-50cd-a5f1-f65ebb7105c5,1
3833,fff4e5f0-4ee5-5fe7-aa30-e870edaf6ed7,2
3834,fff84c0e-90a1-59d8-9997-adc909d50e16,1
3835,fffc17a7-f935-5d3e-bd3e-d761fd80d479,2


In [23]:
df_ctr_links=pd.merge(left=link_clicks,right=link_pageviews,on='linkid', how='inner')

In [33]:
df_ctr_links['ctr']=round(df_ctr_links['clicks']/df_ctr_links['page_views'],2)

In [34]:
df_ctr_links

,linkid,clicks,page_views,ctr
0,00126b32-0c35-507b-981c-02c80d2aa8e7,2,2,1.00
1,004b9724-abca-5481-b6e9-6148a7ca00a5,1,1,1.00
2,0063a982-41cd-5629-96d0-e1c4dd72ea11,2,3,0.67
3,006af6a0-1f0d-4b0c-93bf-756af9071c06,8,36,0.22
4,00759b81-3f04-4a61-b934-f8fb3185f4a0,3,4,0.75
...,...,...,...,...
2248,ffd8d5a7-91bc-48e1-a692-c26fca8a8ead,29,84,0.35
2249,fff38ca0-8043-50cd-a5f1-f65ebb7105c5,1,1,1.00
2250,fff84c0e-90a1-59d8-9997-adc909d50e16,1,1,1.00
2251,fffc17a7-f935-5d3e-bd3e-d761fd80d479,1,2,0.50


## Question 5
### [Pandas] How does the clickrate distribute across different links?

In [35]:
df_ctr_links['ctr'].describe()

count    2253.000000
mean        0.809907
std         1.958050
min         0.090000
25%         0.500000
50%         1.000000
75%         1.000000
max        92.300000
Name: ctr, dtype: float64

## Question 6
### [Pandas & SciPy] Is there any correlation between clicks and previews on a link? Is it significant? How large is the effect? Make sure to at least test for potential linear as well as categorical (think binary) relationships between both variables.